# Delta Lake on MinIO with a catalog
Run the cells in order. This notebook registers Delta tables in the Hive Metastore and uses catalog-qualified table names instead of direct `s3a://` paths.

In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (SparkSession.builder.appName('delta-catalog-notebook')
    .master('spark://spark-master:7077')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.hive.metastore.uris', 'thrift://hive-metastore:9083')
    .config('spark.sql.warehouse.dir', 's3a://warehouse/spark-warehouse')
    .config('spark.jars.ivy', '/tmp/.ivy2')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.secret.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.endpoint.region', 'us-east-1')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem'))
spark = configure_spark_with_delta_pip(
    builder,
).enableHiveSupport().getOrCreate()

## Create a catalog database
The Hive Metastore is the catalog service. `spark_catalog` is configured to use Delta, and the database location tells Spark where managed table data belongs in MinIO.

In [2]:
assert spark._jvm.org.apache.hadoop.fs.s3a.S3AFileSystem is not None, 'S3A connector is unavailable; rebuild the Jupyter image and restart the kernel.'
spark.sql("CREATE DATABASE IF NOT EXISTS notebook_catalog LOCATION 's3a://warehouse/notebook_catalog'")
spark.sql("USE notebook_catalog")
spark.sql("SHOW DATABASES").show()

+----------------+
|       namespace|
+----------------+
|         default|
|notebook_catalog|
+----------------+



## Write, append, and query a registered table

In [3]:
table_name = 'notebook_catalog.people'
df = spark.createDataFrame([(1, 'Ada', 'engineering'), (2, 'Grace', 'data')], ['id', 'name', 'team'])
df.write.format('delta').mode('overwrite').saveAsTable(table_name)
df2 = spark.createDataFrame([(3, 'Lin', 'platform')], ['id', 'name', 'team'])
df2.write.format('delta').mode('append').saveAsTable(table_name)
spark.table(table_name).orderBy('id').show()

+---+-----+-----------+
| id| name|       team|
+---+-----+-----------+
|  1|  Ada|engineering|
|  2|Grace|       data|
|  3|  Lin|   platform|
+---+-----+-----------+



In [4]:
spark.sql("SHOW TABLES IN notebook_catalog").show()
spark.sql("DESCRIBE EXTENDED notebook_catalog.people").filter("col_name = 'Location'").show(truncate=False)

+----------------+-------------+-----------+
|       namespace|    tableName|isTemporary|
+----------------+-------------+-----------+
|notebook_catalog|       people|      false|
|notebook_catalog|people_schema|      false|
+----------------+-------------+-----------+

+--------+---------------------------------------+-------+
|col_name|data_type                              |comment|
+--------+---------------------------------------+-------+
|Location|s3a://warehouse/notebook_catalog/people|       |
+--------+---------------------------------------+-------+



## Schema evolution

In [5]:
schema_table = 'notebook_catalog.people_schema'
spark.createDataFrame([(1, 'Ada')], ['id', 'name']).write.format('delta').mode('overwrite').saveAsTable(schema_table)
spark.createDataFrame([(2, 'Grace', 'data')], ['id', 'name', 'team']).write.format('delta').option('mergeSchema', 'true').mode('append').saveAsTable(schema_table)
spark.table(schema_table).printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- team: string (nullable = true)



## Transaction history and time travel

In [6]:
spark.sql('DESCRIBE HISTORY notebook_catalog.people').select('version', 'operation').show()
spark.read.format('delta').option('versionAsOf', 0).table(table_name).show()

+-------+--------------------+
|version|           operation|
+-------+--------------------+
|      3|               WRITE|
|      2|CREATE OR REPLACE...|
|      1|               WRITE|
|      0|CREATE OR REPLACE...|
+-------+--------------------+

+---+-----+-----------+
| id| name|       team|
+---+-----+-----------+
|  1|  Ada|engineering|
|  2|Grace|       data|
+---+-----+-----------+



In [7]:
spark.stop()